In [6]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import label_binarize
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from scipy import stats

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
# 1. 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [5]:
# 2. 전처리 함수 정의
def preprocess_data(df, is_train=True):
    # ID 및 타겟 처리
    df_id = df['id']
    target = None
    
    if is_train:
        # Heart Disease 문자열을 숫자(0, 1)로 변환
        le = LabelEncoder()
        target = le.fit_transform(df['Heart Disease'])
        df = df.drop(['id', 'Heart Disease'], axis=1)
    else:
        df = df.drop(['id'], axis=1)

    # --- 사용자 제공 전처리 로직 ---
    # 1. 원-핫 인코딩
    oh_cols = ['Chest pain type', 'EKG results', 'Thallium']
    df_prep = pd.get_dummies(df, columns=oh_cols, drop_first=False, dtype=int)

    # 2. 파생 피처 생성
    # (원본 df를 참조하여 조건절 작성)
    df_prep['silent_danger'] = ((df['Chest pain type'] == 4) & (df['Thallium'] == 7)).astype(int)
    df_prep['vessel_st_risk'] = (df['Number of vessels fluro'] * df['ST depression'])
    df_prep['is_senior'] = (df['Age'] >= 60).astype(int)
    
    return df_prep, target, df_id

# 데이터 변환 실행
X, y, _ = preprocess_data(train, is_train=True)
X_test, _, test_id = preprocess_data(test, is_train=False)

In [7]:
# 3. Optuna를 이용한 LightGBM 파라미터 튜닝
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'num_leaves': trial.suggest_int('num_leaves', 31, 128),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    # 3-Fold 교차 검증
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    auc_scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        model = LGBMClassifier(**param)
        model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, preds))
    
    return np.mean(auc_scores)

print("🔍 Optuna 최적화 시작...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) # 시간 관계상 10회, 필요시 30~50회로 늘리세요

print(f"✅ 최적 파라미터: {study.best_params}")


[I 2026-02-23 17:12:34,647] A new study created in memory with name: no-name-7afe5393-170b-49c2-817d-e7c1ffb0bab7


🔍 Optuna 최적화 시작...


[I 2026-02-23 17:12:58,331] Trial 0 finished with value: 0.9547240704982585 and parameters: {'n_estimators': 587, 'learning_rate': 0.016740309799443878, 'max_depth': 11, 'num_leaves': 100, 'subsample': 0.8689804952387066, 'colsample_bytree': 0.8998264618308995}. Best is trial 0 with value: 0.9547240704982585.
[I 2026-02-23 17:13:15,850] Trial 1 finished with value: 0.9551682658103394 and parameters: {'n_estimators': 831, 'learning_rate': 0.05672952811822569, 'max_depth': 5, 'num_leaves': 82, 'subsample': 0.7701008210895988, 'colsample_bytree': 0.7935116084199112}. Best is trial 1 with value: 0.9551682658103394.
[I 2026-02-23 17:13:49,185] Trial 2 finished with value: 0.9546103622557491 and parameters: {'n_estimators': 1001, 'learning_rate': 0.0323058804181151, 'max_depth': 8, 'num_leaves': 112, 'subsample': 0.8496997657389794, 'colsample_bytree': 0.8367055411873024}. Best is trial 1 with value: 0.9551682658103394.
[I 2026-02-23 17:14:29,318] Trial 3 finished with value: 0.9549828967895

✅ 최적 파라미터: {'n_estimators': 831, 'learning_rate': 0.05672952811822569, 'max_depth': 5, 'num_leaves': 82, 'subsample': 0.7701008210895988, 'colsample_bytree': 0.7935116084199112}


In [8]:
# 4. 여러 모델 학습 및 검증
# 최적화된 LGBM + 기본 XGBoost + Random Forest
final_models = {
    "LGBM_Optimized": LGBMClassifier(**study.best_params, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
}

# 8:2 분할로 최종 검증 점수 확인
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

ensemble_preds = []

for name, model in final_models.items():
    model.fit(X_train, y_train)
    val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    print(f"📊 {name} Validation ROC-AUC: {val_auc:.4f}")
    
    # 테스트 데이터 예측 (확률값)
    test_prob = model.predict_proba(X_test)[:, 1]
    ensemble_preds.append(test_prob)

# 5. 최종 앙상블 (Soft Voting: 각 모델의 확률 평균)
# 단순히 한 모델만 쓰는 것보다 일반적으로 AUC가 더 잘 나옵니다.
final_prob = np.mean(ensemble_preds, axis=0)

📊 LGBM_Optimized Validation ROC-AUC: 0.9561
📊 XGBoost Validation ROC-AUC: 0.9558
📊 RandomForest Validation ROC-AUC: 0.9511


In [9]:
# 6. 제출 파일 생성
submission = pd.DataFrame({
    'id': test_id,
    'Heart Disease': final_prob
})

submission.to_csv('submission_final.csv', index=False)
print("🚀 모든 과정 완료! 'submission_final.csv'를 확인하세요.")

submission.head()

🚀 모든 과정 완료! 'submission_final.csv'를 확인하세요.


,id,Heart Disease
0,630000,0.914883
1,630001,0.012235
2,630002,0.978897
3,630003,0.008528
4,630004,0.244329
